# 01 · comma2k19 preprocessing smoke test

Processes only **two segments** from the first archive. Inspect synchronization before processing the full dataset.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, sys, subprocess

DRIVE_ROOT = Path('/content/drive/MyDrive/Blackbox-Detection')
REPO = Path('/content/Blackbox-Detection')
DATA_ROOT = DRIVE_ROOT / 'DATASET'
COMMA_ROOT = DATA_ROOT / 'comma2k19'
RAW_ROOT = COMMA_ROOT / 'raw'
PROCESSED_ROOT = COMMA_ROOT / 'processed' / 'v1'
MANIFEST_ROOT = DRIVE_ROOT / 'manifests' / 'stage3' / 'v1'
OUTPUT_ROOT = DRIVE_ROOT / 'outputs' / 'stage3'
PRETRAINED_ROOT = DRIVE_ROOT / 'pretrained'

# Clone your repository if this runtime does not have it yet.
if not REPO.exists():
    raise RuntimeError('Clone Blackbox-Detection to /content/Blackbox-Detection first, then rerun this cell.')
if str(REPO / 'src') not in sys.path:
    sys.path.insert(0, str(REPO / 'src'))

for p in [PROCESSED_ROOT, MANIFEST_ROOT, OUTPUT_ROOT, PRETRAINED_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

print('RAW_ROOT      :', RAW_ROOT)
print('PROCESSED_ROOT:', PROCESSED_ROOT)
# Install this repository through its existing pyproject.toml without replacing
# Colab's binary stack. Dependency versions in pyproject.toml are aligned to the
# DACON evaluation-server package list.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", str(REPO)],
    check=True,
)


In [ ]:
import pandas as pd
from blackbox_detection.stage3.comma2k19 import find_archives, PrepareConfig, prepare_archive

archives = find_archives(RAW_ROOT)
cfg = PrepareConfig(processed_root=PROCESSED_ROOT, overwrite=False)
report = prepare_archive(archives[0], cfg, max_segments=2)
display(report)
assert 'error' not in report.columns or report['error'].isna().all(), report

In [ ]:
import matplotlib.pyplot as plt
from blackbox_detection.stage3.schema import read_frame_table

row = report.iloc[0]
meta = read_frame_table(PROCESSED_ROOT / row.metadata_relpath)
print(meta.shape)
display(meta.head())

fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
axes[0].plot(meta.timestamp - meta.timestamp.iloc[0], meta.speed_mps); axes[0].set_ylabel('speed m/s')
axes[1].plot(meta.timestamp - meta.timestamp.iloc[0], meta.accel_from_speed_mps2); axes[1].set_ylabel('accel m/s²')
axes[2].plot(meta.timestamp - meta.timestamp.iloc[0], meta.steering_deg); axes[2].set_ylabel('steer deg')
axes[3].plot(meta.timestamp - meta.timestamp.iloc[0], meta.yaw_rate_rps); axes[3].set_ylabel('yaw rad/s'); axes[3].set_xlabel('seconds')
plt.show()

In [ ]:
import cv2, matplotlib.pyplot as plt

video_path = PROCESSED_ROOT / row.video_relpath
cap = cv2.VideoCapture(str(video_path))
print('frames:', int(cap.get(cv2.CAP_PROP_FRAME_COUNT)), 'fps:', cap.get(cv2.CAP_PROP_FPS))
for idx in [0, 150, 300, 450, 599]:
    cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
    ok, bgr = cap.read()
    if not ok: continue
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(8,5)); plt.imshow(rgb); plt.title(f'frame {idx} | speed={meta.speed_mps.iloc[min(idx,len(meta)-1)]:.1f} m/s | steer={meta.steering_deg.iloc[min(idx,len(meta)-1)]:.1f}°'); plt.axis('off'); plt.show()
cap.release()

**Stop here and inspect.** The video should be 10 Hz and metadata should have roughly the same number of rows (normally ~600). Steering/yaw changes should be temporally plausible. Only after this check should notebook 02 be run.